In [1]:
# =========================
# STEP 0 — IMPORTS, PATHS, FOLDERS
# =========================
import os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

try:
    from IPython.display import display
except ImportError:
    def display(x): print(x.head() if hasattr(x,"head") else x)

# Paths
DATA_PATH   = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Data\raw\diabetic_data.csv"
EDA_DIR     = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\eda_visualizations"
OUTPUTS_DIR = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs"

os.makedirs(EDA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("EDA visualizations ->", EDA_DIR)
print("Outputs ->", OUTPUTS_DIR)

# Common config
TARGET_COL   = "readmitted"
DERIVED_TGT  = "readmitted_30d"
ID_COLS      = ["encounter_id","patient_nbr"]
PLACEHOLDERS = ["?","Unknown/Invalid","Unknown","None","N/A","NA","NULL","Not Available"]


EDA visualizations -> C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\eda_visualizations
Outputs -> C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs


In [3]:
# =========================
# STEP 1 — LOAD RAW DATA
# =========================
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Dataset not found at "+DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
display(df.head())

# Replace placeholder missings with NaN
df = df.replace(PLACEHOLDERS, np.nan)
for c in df.select_dtypes(include=["object"]).columns:
    df[c] = df[c].astype(str).str.strip()

Raw shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
# =========================
# STEP 2 — HANDLING MISSING DATA
# =========================
def handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Drop columns with > 40% missing
    thresh = int(len(out) * 0.6)
    out = out.dropna(axis=1, thresh=thresh)
    # Fill numeric NaNs with median, categorical with mode
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(out[col].median())
        else:
            out[col] = out[col].fillna(out[col].mode()[0] if not out[col].mode().empty else "Unknown")
    return out

df = handle_missing(df)
print("After missing data handling:", df.shape)

After missing data handling: (101766, 50)


In [5]:
# =========================
# STEP 3 — ENCODING CATEGORICAL VARIABLES
# =========================
def encode_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    id_like = [c for c in ID_COLS if c in df.columns]
    work = df.drop(columns=id_like) if id_like else df
    cat_cols = work.select_dtypes(include=["object"]).columns.tolist()
    if TARGET_COL in cat_cols: cat_cols.remove(TARGET_COL)
    out = pd.get_dummies(work, columns=cat_cols, drop_first=True)
    if id_like: out = pd.concat([df[id_like], out], axis=1)
    return out

df = encode_categoricals(df)
print("After encoding:", df.shape)

After encoding: (101766, 2437)


In [6]:
# =========================
# STEP 4 — OUTLIER REMOVAL (IQR)
# =========================
def remove_outliers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        q1, q3 = out[col].quantile(0.25), out[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
        out = out[(out[col] >= lower) & (out[col] <= upper)]
    return out.reset_index(drop=True)

df = remove_outliers(df)
print("After outlier removal:", df.shape)

After outlier removal: (56269, 2437)


In [7]:
# =========================
# STEP 5 — FEATURE ENGINEERING
# =========================
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    fe_df = df.copy()
    # Example: age midpoint from bracket
    if "age" in fe_df.columns and fe_df["age"].dtype == "object":
        def age_mid(x):
            nums = re.findall(r"\d+", str(x))
            return (int(nums[0])+int(nums[1]))/2 if len(nums)>=2 else np.nan
        fe_df["fe_age_mid"] = fe_df["age"].apply(age_mid)
    # Example: total visits
    for need in ["number_outpatient","number_emergency","number_inpatient"]:
        if need not in fe_df.columns:
            fe_df[need] = 0
    fe_df["fe_total_visits"] = (
        fe_df["number_outpatient"] + fe_df["number_emergency"] + fe_df["number_inpatient"]
    )
    return fe_df

df = feature_engineering(df)
print("After feature engineering:", df.shape)

After feature engineering: (56269, 2438)


In [8]:
# =========================
# STEP 6 — NORMALIZATION
# =========================
def normalize_numeric(df: pd.DataFrame, method="standard") -> pd.DataFrame:
    exclude = set([TARGET_COL, DERIVED_TGT] + [c for c in ID_COLS if c in df.columns])
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
    out = df.copy()
    for c in num_cols:
        if out[c].isna().any():
            out[c] = out[c].fillna(out[c].median())
    if method=="standard":
        scaler = StandardScaler()
    elif method=="minmax":
        scaler = MinMaxScaler()
    else:
        scaler = RobustScaler()
    if num_cols:
        out[num_cols] = scaler.fit_transform(out[num_cols])
    return out

df = normalize_numeric(df, method="standard")
print("After normalization:", df.shape)

After normalization: (56269, 2438)


In [ ]:
# =========================
# STEP 7 — FINAL CLEANUP & SAVE
# =========================
def final_clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.drop_duplicates().reset_index(drop=True)
    if "readmitted" in out.columns and "readmitted_30d" not in out.columns:
        out["readmitted_30d"] = out["readmitted"].map({"<30":1, ">30":0, "NO":0})
    return out

df = final_clean(df)

final_path = os.path.join(OUTPUTS_DIR,"final_processed_diabetic.csv")
df.to_csv(final_path, index=False)
print("Final dataset saved to:", final_path)
display(df.head())